# Week 1 - Data Understanding & Preparation

**Project:** Excelerate Opportunity Dataset

**Objectives**
- Load and inspect the dataset
- Understand dataset structure
- Create a data quality assessment
- Clean the dataset
- Validate the cleaned dataset
- Export an analysis-ready dataset


In [ ]:
import pandas as pd
import numpy as np

# Update the path if needed
file_path = "#OpportunityData.xlsx"

df = pd.read_excel(file_path)
print("Dataset loaded successfully!")


## 1. Dataset Overview

In [ ]:
print("Rows, Columns:", df.shape)
display(df.head())
display(df.info())
display(df.describe(include='all').T)


## 2. Column Information

In [ ]:
column_info = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str),
    "Missing Values": df.isna().sum(),
    "Unique Values": [df[c].nunique(dropna=True) for c in df.columns]
})
display(column_info)
column_info.to_excel("Data_Dictionary_Base.xlsx", index=False)


## 3. Missing Value Analysis

In [ ]:
missing = pd.DataFrame({
    "Missing Count": df.isna().sum(),
    "Missing %": (df.isna().sum()/len(df)*100).round(2)
}).sort_values("Missing %", ascending=False)

display(missing)


## 4. Duplicate Check

In [ ]:
print("Duplicate Rows:", df.duplicated().sum())

if "opportunity_id" in df.columns:
    print("Duplicate opportunity_id:",
          df["opportunity_id"].duplicated().sum())


## 5. Data Cleaning

In [ ]:
clean_df = df.copy()

# Standardize column names
clean_df.columns = (
    clean_df.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

# Remove duplicate rows
clean_df = clean_df.drop_duplicates()

# Trim whitespace
for col in clean_df.select_dtypes(include="object").columns:
    clean_df[col] = clean_df[col].astype(str).str.strip()

# Convert numeric columns if present
for col in ["duration","fee","microscholarship"]:
    if col in clean_df.columns:
        clean_df[col] = pd.to_numeric(clean_df[col], errors="coerce")

# Convert Unix timestamp columns
for col in ["created_at","modified_at","last_date_to_apply"]:
    if col in clean_df.columns:
        clean_df[col] = pd.to_numeric(clean_df[col], errors="coerce")
        clean_df[col] = pd.to_datetime(clean_df[col], unit="ms", errors="coerce")

print(clean_df.info())


## 6. Validation

In [ ]:
print("Cleaned Shape:", clean_df.shape)
print("Remaining Duplicate Rows:", clean_df.duplicated().sum())

validation = pd.DataFrame({
    "Column": clean_df.columns,
    "Missing Values": clean_df.isna().sum()
})

display(validation)


## 7. Export Cleaned Dataset

In [ ]:
clean_df.to_excel("Cleaned_Opportunity_Dataset.xlsx", index=False)
clean_df.to_csv("Cleaned_Opportunity_Dataset.csv", index=False)

print("Cleaned dataset exported successfully.")
